# PPO on SuperAutoStrat

The reusable pieces live in `agents/ppo/`:

| file | what |
|---|---|
| `config.py` | `PPOConfig` hyperparameters, `get_device()` |
| `network.py` | `ActorCritic`: shared trunk, policy logits + value head, masked sampling |
| `envs.py` | `make_vec_env`: N parallel envs (the actors) |
| `rollout.py` | `collect_rollout`: run pi_old for T steps |
| `buffer.py` | `RolloutBuffer`: (T x N) storage + minibatches |
| `advantages.py` | **`compute_gae` is yours to write** (`pytest tests/test_ppo.py` checks it) |
| `losses.py` | `ppo_clip_loss`; **`value_loss`, `entropy_bonus` are yours** |
| `tracking.py` | `EpisodeTracker` for logging |

In [1]:
%load_ext autoreload
%autoreload 2

import torch
from torch import nn

from agents.ppo import (
    ActorCritic, EpisodeTracker, PPOConfig, RolloutBuffer, collect_rollout, compute_gae,
    entropy_bonus, get_device, make_vec_env, ppo_clip_loss, value_loss,
)

cfg = PPOConfig()
device = get_device()
print(f"Using {device} device")

Using cpu device


# Parallel actors

`N = cfg.n_envs` environments, each in its own process, stepped together. `obs` is `(N, n_states)` and `mask` is `(N, n_actions)`.

In [2]:
envs, obs, mask = make_vec_env(cfg.n_envs, async_envs=cfg.async_envs, seed=0)

n_states = envs.single_observation_space.shape[0]
n_actions = int(envs.single_action_space.n)
obs.shape, mask.shape

((8, 1146), (8, 132))

# Make the Policy Exist

In [3]:
model = ActorCritic(n_states, n_actions).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, eps=1e-5)

# Make the Surrogate Objectives Exist

Sanity check that `ppo_clip_loss` works as an ordinary loss for Adam: build a loss on a fake batch, backprop, step, and watch it go down. (Advantages are random here; real ones come from GAE.)

$$
L_{t}^{CLIP} (\theta) = \min (r_t(\theta)\cdot \hat{A}, \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon)\cdot \hat{A}) \\

L_{t}^{VF} = (V_{\theta} (s_t) - V_{t}^{targ})^2 \\

S[\pi_{\theta}](s_t) = \text{Entropy bonus} \\

L^{CLIP + VF + S} = \mathbb{\hat{E}_t} [L_{t}^{CLIP} -c_1 L_{t}^{VF} + c_2 S[\pi_{\theta}](s_t)] 
$$

In [4]:
model = ActorCritic(n_states, n_actions).to(device)  # throw away the fake update
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, eps=1e-5)

# Run Proximal Policy Optimization

In [5]:
buffer = RolloutBuffer(cfg.n_steps, cfg.n_envs, n_states, n_actions, device)
tracker = EpisodeTracker(cfg.n_envs)

for update in range(cfg.n_updates):
    # 1. Run pi_old in N envs for T steps
    obs, mask, last_value = collect_rollout(envs, model, buffer, obs, mask, device, tracker)

    # 2. Compute advantage estimates A_1..A_T
    buffer.advantages, buffer.returns = compute_gae(
        buffer.rewards, buffer.values, buffer.dones, last_value, cfg.gamma, cfg.gae_lambda
    )

    # 3. Optimise surrogate L wrt theta, K epochs, minibatch size M <= N*T
    for epoch in range(cfg.n_epochs):
        for mb in buffer.minibatches(cfg.minibatch_size):
            dist, new_values = model.distribution(mb["obs"], mb["masks"])   # one forward pass, both heads
            new_log_probs = dist.log_prob(mb["actions"])

            loss = (
                ppo_clip_loss(new_log_probs, mb["log_probs"], mb["advantages"], cfg.clip_eps)
                + cfg.value_coef * value_loss(new_values, mb["returns"])
                - cfg.entropy_coef * entropy_bonus(dist.entropy())
            )
            
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            # 4. Theta update
            optimizer.step()

    episodes = tracker.pop()
    if episodes:
        rets, lens = [e.ret for e in episodes], [e.length for e in episodes]
        print(f"update {update}: {len(episodes)} episodes, "
              f"mean return {sum(rets)/len(rets):.2f}, mean length {sum(lens)/len(lens):.0f}")

update 0: 4 episodes, mean return -2.25, mean length 90
update 1: 12 episodes, mean return 1.00, mean length 112
update 2: 5 episodes, mean return -0.80, mean length 100
update 3: 7 episodes, mean return 2.57, mean length 160
update 4: 8 episodes, mean return 2.00, mean length 132
update 5: 6 episodes, mean return 4.50, mean length 162
update 6: 7 episodes, mean return 4.14, mean length 164
update 7: 7 episodes, mean return 3.57, mean length 176
update 8: 4 episodes, mean return 3.50, mean length 161
update 9: 5 episodes, mean return 2.80, mean length 176
update 10: 7 episodes, mean return 5.43, mean length 181
update 11: 7 episodes, mean return 7.43, mean length 165
update 12: 6 episodes, mean return 4.00, mean length 146
update 13: 6 episodes, mean return 4.17, mean length 170
update 14: 9 episodes, mean return 4.22, mean length 147
update 15: 6 episodes, mean return 5.17, mean length 149
update 16: 7 episodes, mean return 3.86, mean length 139
update 17: 7 episodes, mean return 6.57

c:\Users\rcorr\anaconda3\envs\quant\Lib\site-packages\gymnasium\vector\async_vector_env.py:462: UserWarning: ERROR: Received the following error from Worker-3 - Shutting it down
  self._raise_if_errors(successes)
c:\Users\rcorr\anaconda3\envs\quant\Lib\site-packages\gymnasium\vector\async_vector_env.py:462: UserWarning: ERROR: Traceback (most recent call last):
  File "c:\Users\rcorr\anaconda3\envs\quant\Lib\site-packages\gymnasium\vector\async_vector_env.py", line 781, in _async_worker
    ) = env.step(data)
        ~~~~~~~~^^^^^^
  File "c:\Users\rcorr\anaconda3\envs\quant\Lib\site-packages\gymnasium\wrappers\common.py", line 392, in step
    return super().step(action)
           ~~~~~~~~~~~~^^^^^^^^
  File "c:\Users\rcorr\anaconda3\envs\quant\Lib\site-packages\gymnasium\core.py", line 327, in step
    return self.env.step(action)
           ~~~~~~~~~~~~~^^^^^^^^
  File "c:\Users\rcorr\anaconda3\envs\quant\Lib\site-packages\gymnasium\wrappers\common.py", line 284, in step
    return

KeyError: KeyError('Bee')

In [ ]:
# envs.close()

# Tracked training

The loop above, packaged as `agents.ppo.train` with tracking. Each run gets:

| where | what |
|---|---|
| `runs/optuna.db` | one Optuna trial per run: hyperparameters, best eval win rate, state (COMPLETE / FAIL / PRUNED) |
| `runs/<run>/tb/` | TensorBoard curves: `train/*`, `eval/*`, `loss/*` (incl. approx KL, clip fraction, explained variance) |
| `runs/<run>/metrics.jsonl` | the same numbers, one line per update (for pandas) |
| `runs/<run>/replays/update_XXXX.jsonl` | every evaluation game, decision by decision |
| `runs/<run>/checkpoints/` | `latest.pt`, `best.pt`, `update_XXXX.pt` |

Live curves while it trains: `tensorboard --logdir runs`. Compare runs: `optuna-dashboard sqlite:///runs/optuna.db`.

In [ ]:
from agents.ppo import train, load_runs, load_metrics, load_replays

cfg = PPOConfig(n_updates=300, eval_every=25, eval_episodes=20)
result = train(cfg, "baseline", log_every=10)
result.run_dir, result.best_eval_win_rate

# How do runs compare? (question 3)

In [ ]:
runs = load_runs()
cols = ["number", "state", "value", "user_attrs_run_name", "params_lr", "params_entropy_coef",
        "params_n_updates", "user_attrs_best_update", "duration"]
runs[[c for c in cols if c in runs.columns]].sort_values("value", ascending=False)

# How did it play? (question 1)

Training curves of a run, then the recorded evaluation games.

In [ ]:
import matplotlib.pyplot as plt

m = load_metrics(result.run_dir)
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
panels = [
    ("win rate", ["train/win_rate", "eval/win_rate"]),
    ("wins per game", ["train/wins", "eval/wins"]),
    ("episode return", ["train/ret", "eval/ret"]),
    ("entropy", ["loss/entropy"]),
    ("approx KL / clip frac", ["loss/approx_kl", "loss/clip_frac"]),
    ("explained variance", ["loss/explained_var"]),
]
for ax, (title, keys) in zip(axes.flat, panels):
    for k in keys:
        if k in m:
            s = m[k].dropna()
            ax.plot(s.index, s.values, marker="o" if k.startswith("eval") else None, ms=3, label=k)
    ax.set_title(title); ax.set_xlabel("update"); ax.legend(fontsize=8)
fig.tight_layout()

In [ ]:
from agents.ppo.replay import latest_replay, show_episode, summarize_file

records = load_replays(latest_replay(result.run_dir))
print(summarize_file(records))

In [ ]:
# Step through one recorded game: one frame per round (battles_only), or every decision.
show_episode(records[0], battles_only=True)

# How is it playing? (question 2)

Plays a fresh game with the saved policy and redraws the board each decision: state, last action and battle, and the policy's top actions with V(s). The same view in a terminal: `python -m agents.ppo.watch runs/baseline --delay 0.5`.

In [ ]:
from agents.ppo.watch import watch_in_notebook

_ = watch_in_notebook(result.run_dir, delay=0.3, seed=0)